# Model Training & Data Preprocessing
This notebook covers the steps for data loading, preprocessing, model building, training, and saving the trained image captioning model.

In [ ]:
# Install required libraries
!pip install tensorflow==2.15.0  # Ensure correct TensorFlow version

In [ ]:
# Import necessary libraries for model building and training
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout, Add, Bidirectional
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
import numpy as np
import os
import pickle  # For loading and saving Python objects

## Load the Data

In [ ]:
# Load image features, captions, and max_length from preprocessed files
import os
required_paths = ['dataset/image_features.pkl','dataset/captions.pkl','dataset/max_length.pkl']
for p in required_paths:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Required file not found: {p} — please place it in the `dataset/` folder")

with open('dataset/image_features.pkl','rb') as a:
    features = pickle.load(a)
with open('dataset/captions.pkl','rb') as b:
    mapping = pickle.load(b)
with open('dataset/max_length.pkl','rb') as c:
    max_length = pickle.load(c)

# Basic sanity checks
if not isinstance(mapping, dict):
    raise TypeError('`captions.pkl` should contain a dict mapping image ids to captions')
if not hasattr(features, '__getitem__'):
    raise TypeError('`image_features.pkl` should be indexable (dict-like)')

print(f"Loaded {len(mapping)} image caption entries and {len(features)} image feature entries. max_length={max_length}")

## Tokenizing Text

In [ ]:
# Add start and end tokens to each caption for sequence modeling
for image_id, captions in mapping.items():
    for idx in range(len(captions)):
        captions[idx] = f'<start> {captions[idx]} <end>'

# Collect all captions into a single list for tokenization
all_captions = [caption for captions in mapping.values() for caption in captions]

# Tokenize captions to convert words to integer sequences
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_captions)
vocab_size = len(tokenizer.word_index) + 1  # Total number of unique words (+1 for padding)

# Save tokenizer and vocabulary size for later use
os.makedirs('dataset', exist_ok=True)
with open('dataset/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
with open('dataset/vocab_size.pkl', 'wb') as f:
    pickle.dump(vocab_size, f)

print(f"Tokenizer fitted: vocab_size={vocab_size}")

## Train/Test Split

In [ ]:
# Split image IDs into training and testing sets (80/20 split)
image_ids = list(mapping.keys())
split = int(len(image_ids) * 0.8)
train = image_ids[:split]  # Training set
test = image_ids[split:]   # Testing set

## Data Generator

In [ ]:
def data_generator(data_keys, mapping, features, tokenizer, max_length, vocab_size, batch_size):
    """Yield batches of ([image_features, input_sequence], target_word_one_hot).

    Handles a few possible shapes for stored image features (list, ndarray, nested arrays).
    """
    while True:
        X1, X2, y = [], [], []
        for key in data_keys:
            captions = mapping.get(key, [])
            if key not in features:
                # Skip missing feature entry with a warning
                print(f"Warning: feature for key {key} not found — skipping")
                continue
            feature = features[key]
            # Normalize feature to 1D feature vector
            if isinstance(feature, (list, tuple)):
                feature = np.array(feature)
            if isinstance(feature, np.ndarray) and feature.ndim > 1:
                # Some saved formats store an extra leading dim
                feature_vec = feature[0]
            else:
                feature_vec = np.array(feature)

            for caption in captions:
                sequence = tokenizer.texts_to_sequences([caption])[0]
                for i in range(1, len(sequence)):
                    in_seq, out_seq = sequence[:i], sequence[i]
                    in_seq = pad_sequences([in_seq], maxlen=max_length, padding='pre')[0]
                    out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]
                    X1.append(feature_vec)
                    X2.append(in_seq)
                    y.append(out_seq)
                    if len(X1) == batch_size:
                        yield [np.array(X1), np.array(X2)], np.array(y)
                        X1, X2, y = [], [], []
        # Yield remaining samples if any
        if len(X1) > 0:
            yield [np.array(X1), np.array(X2)], np.array(y)
            X1, X2, y = [], [], []

## Build the Model

In [ ]:
# Build the image captioning model using a sensible feature vector size inferred from `features`
# Infer feature vector dimensionality from the first entry in `features`
example_feature = next(iter(features.values()))
if isinstance(example_feature, np.ndarray) and example_feature.ndim > 1:
    feature_dim = example_feature.shape[1]
else:
    feature_dim = np.array(example_feature).shape[0]

print(f"Using feature vector dimension = {feature_dim}")

# Encoder: Processes image features
inputs1 = Input(shape=(feature_dim,))
fe1 = Dropout(0.5)(inputs1)
fe2 = Dense(512, activation='relu')(fe1)
# Decoder: Processes caption sequences
inputs2 = Input(shape=(max_length,))
se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
se2 = Dropout(0.5)(se1)
se3 = Bidirectional(LSTM(256))(se2)
# Combine encoder and decoder outputs
decoder1 = Add()([fe2, se3])
decoder2 = Dense(256, activation='relu')(decoder1)
outputs = Dense(vocab_size, activation='softmax')(decoder2)
# Define and compile the model
model = Model(inputs=[inputs1, inputs2], outputs=outputs)
model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=0.001), metrics=['accuracy'])
model.summary()  # Print model architecture

## Train the Model

In [ ]:
# Train the model using the data generator with practical callbacks and safety checks
batch_size = 64
steps_per_epoch = max(1, len(train) // batch_size)
epochs = 50

# Ensure models directory exists
os.makedirs('models', exist_ok=True)

generator = data_generator(train, mapping, features, tokenizer, max_length, vocab_size, batch_size)

# Callbacks: save best model, reduce LR on plateau, early stopping
checkpoint = ModelCheckpoint(os.path.join('models','model_best.h5'), save_best_only=True, monitor='loss', mode='min')
reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.2, patience=5, min_lr=1e-6)
early_stop = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

try:
    model.fit(
        generator,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        verbose=1,
        callbacks=[checkpoint, reduce_lr, early_stop],
        use_multiprocessing=False,
        workers=1,
        max_queue_size=10
    )
except Exception as e:
    print('Training failed with exception:', e)
    print('You may need to reduce batch_size or check data shapes')

# Save the final model (also saved by checkpoint when improved)
model.save(os.path.join('models','model_final.h5'))
print('Final model saved to models/model_final.h5')

## Save the Model

In [ ]:
# Save the trained model for later usage (ensure models folder exists)
import os
os.makedirs('models', exist_ok=True)
model.save(os.path.join('models','model_final.h5'))
print('Model saved to models/model_final.h5')